###  Image processing

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install --upgrade --no-deps tensorflow==2.17.0 opencv-python==4.9.0.80 numpy==1.26.4 pydicom nibabel dicompyler-core matplotlib scikit-learn seaborn tqdm

In [ ]:
!pip install -q pydicom==2.3.1 rt-utils SimpleITK


In [ ]:
import numpy as np
from collections import defaultdict
import pandas as pd
import tensorflow as tf
import cv2
import os
import pydicom
import SimpleITK as sitk
from rt_utils import RTStructBuilder
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import nibabel as nib
from ipywidgets import interact, IntSlider
from tqdm import tqdm


print("Pydicom:", pydicom.__version__)
print("SimpleITK:", sitk.__version__)
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)

**Organizing image folders**

```
NSCLC/NSCLC-Radiomics/
    LUNG1-001/
        09-18-2008-StudyID-NA-69331/
            0.000000-NA-82046/ <-- Reconstructed CT with lung filter (high resolution, thin slices)
            3.000000-NA-78236/ <-- Contains patient contours or ROI, usually drawn in radiotherapy software. It is not a series of images, but structures superimposed on the CT.
            300.000000-Segmentation-9.554/ <-- Segmentation mask (SEG) - Defines the tumor region
```

Each patient has a LUNG1-XXX file.

Within it, each study has a folder with the study date.

DICOM: Image data

Computed Tomography (CT): A set of axial slices representing the patient's anatomy. Each CT slice is 3 mm thick.

Segmentation (SEG): Masks that define regions of interest.
Radiotherapy Structures (RTSTRUCT): Outlines drawn by physicians for radiotherapy planning. Associated with a specific CT series.

In [ ]:
# Check the image data types

base_dir = "/content/drive/MyDrive/NSCLC/NSCLC-Radiomics/LUNG1-027/01-01-2014-StudyID-NA-35913"

for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if os.path.isdir(folder_path):
        first_file = os.listdir(folder_path)[0]
        dcm_path = os.path.join(folder_path, first_file)
        ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
        print(folder, "→", ds.Modality)
        print("SeriesDescription:", ds.get("SeriesDescription", "N/A"))
        print("ProtocolName:", ds.get("ProtocolName", "N/A"))
        print("SeriesNumber:", ds.get("SeriesNumber", "N/A"))
        print("Contrast/Bolus Agent:", ds.get("ContrastBolusAgent", "N/A"))
        print("Contrast/Bolus Administration Route:", ds.get("ContrastBolusRoute", "N/A"))


Output - Check the image data types

```
300.000000-Segmentation-8.487 → SEG
SeriesDescription: Segmentation
ProtocolName: N/A
SeriesNumber: 300
Contrast/Bolus Agent: N/A
Contrast/Bolus Administration Route: N/A
2.000000-NA-63878 → RTSTRUCT
SeriesDescription: N/A
ProtocolName: N/A
SeriesNumber: 2
Contrast/Bolus Agent: N/A
Contrast/Bolus Administration Route: N/A
1.000000-NA-45865 → CT
SeriesDescription: N/A
ProtocolName: N/A
SeriesNumber: 1
Contrast/Bolus Agent: N/A
Contrast/Bolus Administration Route: N/A
```

#### 1. Format conversion

In [ ]:

# Libraries: os, pydicom, SimpleITK, collections


# Standardized DICOM UIDs
CT_UID = "1.2.840.10008.5.1.4.1.1.2"
SEG_UID = "1.2.840.10008.5.1.4.1.1.66.4"


def find_series(patient_path):
    ct_series = defaultdict(list)
    seg_file = None
    ref_uid_seg = None

    for root, _, files in os.walk(patient_path):
        for f in files:
            if not f.lower().endswith(".dcm"):
                continue

            dcm_path = os.path.join(root, f)
            try:
                # Read only the header
                d = pydicom.dcmread(dcm_path, stop_before_pixels=True)
                sop = d.SOPClassUID

                if sop == CT_UID:
                    series_uid = d.SeriesInstanceUID
                    ct_series[series_uid].append(dcm_path)

                elif sop == SEG_UID:
                    seg_file = dcm_path
                    # Attempts to retrieve the UID referenced by SEG
                    if hasattr(d, 'ReferencedSeriesSequence') and d.ReferencedSeriesSequence:
                        ref_uid_seg = d.ReferencedSeriesSequence[0].SeriesInstanceUID

            except Exception:
                pass

    if not ct_series:
        return None, None

    
    if seg_file and ref_uid_seg and ref_uid_seg in ct_series:
        chosen_series_uid = ref_uid_seg
    # Select the series with the most slices
    elif ct_series:
        chosen_series_uid = max(ct_series, key=lambda k: len(ct_series[k]))
    else:
        return None, None

    chosen_ct_folder = os.path.dirname(ct_series[chosen_series_uid][0])

    return chosen_ct_folder, seg_file

# Load CT as a volume
def load_ct(ct_folder):
    reader = sitk.ImageSeriesReader()
    dicom_files = reader.GetGDCMSeriesFileNames(ct_folder)
    reader.SetFileNames(dicom_files)
    return reader.Execute()

#  Load and align the SEG mask
def load_seg(seg_file, ct_img):
    try:
        # Carrega o volume SEG
        seg = sitk.ReadImage(seg_file)
        seg_resampled = sitk.Resample(
            seg,
            ct_img,
            sitk.Transform(),
            sitk.sitkNearestNeighbor,
            0
        )

        return sitk.Cast(seg_resampled, sitk.sitkUInt8)

    except Exception as e:
        print(f"Error in SEG's final alignment: {e}. Returning an empty mask.")
        empty_mask = sitk.Image(ct_img.GetSize(), sitk.sitkUInt8)
        empty_mask.CopyInformation(ct_img)
        return empty_mask

# Save as NIfTI (.nii.gz)
def save_nii(image, path):
    sitk.WriteImage(image, path)
    print(f"Save: {path}")


root = "/content/drive/MyDrive/NSCLC/NSCLC-Radiomics"
output_folder_name = "nii_aligned"

for patient in sorted(os.listdir(root)):
    patient_path = os.path.join(root, patient)
    if not os.path.isdir(patient_path) or not patient.startswith("LUNG1-"):
        continue

    print(f"\n Patient {patient}")

    output_path = os.path.join(patient_path, output_folder_name)
    os.makedirs(output_path, exist_ok=True)

    ct_folder, seg_file = find_series(patient_path)

    if ct_folder is None:
        print("No CT series found.")
        continue

    if seg_file is None:
        print("No SEG files found.")
        continue

    print(f"Selected CT: {ct_folder.split('/')[-1]}")

    try:
        
        ct_volume = load_ct(ct_folder)

        seg_volume = load_seg(seg_file, ct_volume)

        save_nii(ct_volume, os.path.join(output_path, "ct_aligned.nii.gz"))
        save_nii(seg_volume, os.path.join(output_path, "seg_aligned.nii.gz"))

    except Exception as e:
        print(f"Error while processing {patient}: {e}")